# MaterialFocusNet: Attention-Guided Household Waste Classification

**Student:** Abdullayev Namig  
**Course:** Deep Learning Final Project  

This notebook implements the project described in the approved proposal. It compares a simple CNN baseline with **MaterialFocusNet**, a compact custom CNN containing Squeeze-and-Excitation channel-attention blocks. The task is to classify one household-waste item as cardboard, glass, metal, paper, plastic, or trash.

Run this notebook in Google Colab with a GPU runtime. All reported results must be produced by running the training and evaluation cells; no metrics are pre-filled.

## Project protocol

- **Dataset:** TrashNet (2,527 RGB images, six classes).
- **Split:** reproducible stratified 70% training / 15% validation / 15% testing.
- **Input:** 224 x 224 RGB images.
- **Data augmentation:** random resized crop, horizontal flip, rotation up to 15 degrees, and colour jitter on training data only.
- **Imbalance handling:** inverse-frequency class weights in cross-entropy loss.
- **Baseline:** three-block CNN (32, 64, 128 channels).
- **Custom model:** MaterialFocusNet (32, 64, 128, 192 channels) with Squeeze-and-Excitation attention in blocks 3 and 4.
- **Selection metric:** validation macro F1. The test set is used only once for final evaluation.


In [ ]:
# Run in Colab. The first line confirms that a GPU is attached.
!nvidia-smi
!git clone --depth 1 https://github.com/garythung/trashnet.git
!unzip -q trashnet/data/dataset-resized.zip -d data


In [ ]:
import os
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 60
PATIENCE = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print(f'Using device: {DEVICE}')


In [ ]:
# TrashNet folders are named with class labels.
DATA_DIR = Path('data/dataset-resized')
assert DATA_DIR.exists(), 'Dataset folder not found. Run the download cell first.'

class_names = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
class_to_idx = {name: i for i, name in enumerate(class_names)}
samples = [(path, class_to_idx[path.parent.name]) for path in sorted(DATA_DIR.glob('*/*')) if path.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
labels = np.array([label for _, label in samples])

counts = Counter(labels)
distribution = pd.DataFrame({
    'class': class_names,
    'images': [counts[class_to_idx[name]] for name in class_names]
})
display(distribution)
print(f'Total images: {len(samples)}')


In [ ]:
# Stratified 70 / 15 / 15 split. The second split divides the 30% holdout equally.
indices = np.arange(len(samples))
splitter_1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, holdout_idx = next(splitter_1.split(indices, labels))

splitter_2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_relative, test_relative = next(splitter_2.split(holdout_idx, labels[holdout_idx]))
val_idx, test_idx = holdout_idx[val_relative], holdout_idx[test_relative]

assert set(train_idx).isdisjoint(val_idx) and set(train_idx).isdisjoint(test_idx) and set(val_idx).isdisjoint(test_idx)
print(f'Train: {len(train_idx)} | Validation: {len(val_idx)} | Test: {len(test_idx)}')


In [ ]:
# Compute normalization statistics from the TRAINING split only.
stats_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMAGE_SIZE), transforms.ToTensor()])

class WasteDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        path, label = self.records[index]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

stats_loader = DataLoader(WasteDataset([samples[i] for i in train_idx], stats_transform), batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
channel_sum, channel_sq_sum, pixels = torch.zeros(3), torch.zeros(3), 0
for images, _ in stats_loader:
    channel_sum += images.sum(dim=[0, 2, 3])
    channel_sq_sum += (images ** 2).sum(dim=[0, 2, 3])
    pixels += images.shape[0] * images.shape[2] * images.shape[3]
mean = channel_sum / pixels
std = (channel_sq_sum / pixels - mean ** 2).sqrt()
print('Training-only mean:', mean.tolist())
print('Training-only std: ', std.tolist())


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.80, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.15, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist()),
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist()),
])

train_records = [samples[i] for i in train_idx]
val_records = [samples[i] for i in val_idx]
test_records = [samples[i] for i in test_idx]
train_ds = WasteDataset(train_records, train_transform)
val_ds = WasteDataset(val_records, eval_transform)
test_ds = WasteDataset(test_records, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

train_labels = labels[train_idx]
class_counts = np.bincount(train_labels, minlength=len(class_names))
class_weights = len(train_labels) / (len(class_names) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print(pd.DataFrame({'class': class_names, 'train_count': class_counts, 'loss_weight': class_weights.cpu().numpy()}))


In [ ]:
# Inspect augmented training examples.
images, targets = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, target in zip(axes.flat, images[:8], targets[:8]):
    view = image.permute(1, 2, 0) * std.numpy() + mean.numpy()
    ax.imshow(view.clip(0, 1))
    ax.set_title(class_names[target])
    ax.axis('off')
plt.tight_layout()


## Models

The baseline has three convolutional feature blocks. MaterialFocusNet adds a fourth block with 192 channels and Squeeze-and-Excitation attention after blocks 3 and 4. Both models use the same input resolution and training protocol, so the comparison isolates the impact of the architecture.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32), ConvBlock(32, 64), ConvBlock(64, 128)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.40),
            nn.Linear(128, 96), nn.ReLU(inplace=True), nn.Linear(96, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, hidden, 1), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.gate(x)

class MaterialFocusNet(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.block1 = ConvBlock(3, 32)
        self.block2 = ConvBlock(32, 64)
        self.block3 = nn.Sequential(ConvBlock(64, 128), SqueezeExcitation(128))
        self.block4 = nn.Sequential(ConvBlock(128, 192), SqueezeExcitation(192))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.40),
            nn.Linear(192, 96), nn.ReLU(inplace=True), nn.Linear(96, num_classes)
        )
    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        return self.classifier(x)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for model in [BaselineCNN(), MaterialFocusNet()]:
    print(f'{model.__class__.__name__}: {count_parameters(model):,} trainable parameters')


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss, all_targets, all_predictions = 0.0, [], []
    for images, targets in loader:
        images, targets = images.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, targets)
            if is_training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * images.size(0)
        all_targets.extend(targets.detach().cpu().numpy())
        all_predictions.extend(logits.argmax(dim=1).detach().cpu().numpy())
    macro_f1 = precision_recall_fscore_support(all_targets, all_predictions, average='macro', zero_division=0)[2]
    accuracy = accuracy_score(all_targets, all_predictions)
    return total_loss / len(loader.dataset), accuracy, macro_f1

def train_model(model, name):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
    best_f1, stale_epochs, best_state = -np.inf, 0, None
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_acc, train_f1 = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_f1 = run_epoch(model, val_loader, criterion)
        scheduler.step()
        for key, value in [('train_loss', train_loss), ('val_loss', val_loss), ('train_f1', train_f1), ('val_f1', val_f1)]: history[key].append(value)
        print(f'{name} | epoch {epoch:02d} | train F1 {train_f1:.4f} | val F1 {val_f1:.4f} | val acc {val_acc:.4f}')
        if val_f1 > best_f1:
            best_f1, stale_epochs = val_f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                print(f'Early stopping at epoch {epoch}; best validation macro F1 = {best_f1:.4f}')
                break
    model.load_state_dict(best_state)
    torch.save({'model_state_dict': model.state_dict(), 'best_val_macro_f1': best_f1}, f'{name}_best.pt')
    return model, history, best_f1


In [ ]:
# Training can take several minutes on a Colab GPU. Keep this cell's printed output for the report.
baseline_model, baseline_history, baseline_val_f1 = train_model(BaselineCNN(), 'baseline_cnn')
material_model, material_history, material_val_f1 = train_model(MaterialFocusNet(), 'material_focus_net')


In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='train'); axes[0].plot(history['val_loss'], label='validation')
    axes[0].set_title(f'{title}: loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
    axes[1].plot(history['train_f1'], label='train'); axes[1].plot(history['val_f1'], label='validation')
    axes[1].set_title(f'{title}: macro F1'); axes[1].set_xlabel('epoch'); axes[1].legend()
    plt.tight_layout()

plot_history(baseline_history, 'Baseline CNN')
plot_history(material_history, 'MaterialFocusNet')


In [ ]:
@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for images, targets in loader:
        logits = model(images.to(DEVICE, non_blocking=True))
        y_true.extend(targets.numpy())
        y_pred.extend(logits.argmax(dim=1).cpu().numpy())
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {'accuracy': accuracy_score(y_true, y_pred), 'macro_precision': precision, 'macro_recall': recall, 'macro_f1': f1, 'y_true': y_true, 'y_pred': y_pred}

baseline_test = evaluate_model(baseline_model, test_loader)
material_test = evaluate_model(material_model, test_loader)
results = pd.DataFrame([
    {k: v for k, v in baseline_test.items() if not isinstance(v, list)} | {'model': 'Baseline CNN'},
    {k: v for k, v in material_test.items() if not isinstance(v, list)} | {'model': 'MaterialFocusNet'},
]).set_index('model')
display(results.style.format('{:.4f}'))


In [ ]:
def show_evaluation(result, title):
    print(title)
    print(classification_report(result['y_true'], result['y_pred'], target_names=class_names, zero_division=0))
    cm = confusion_matrix(result['y_true'], result['y_pred'])
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{title} - test confusion matrix'); plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout()

show_evaluation(baseline_test, 'Baseline CNN')
show_evaluation(material_test, 'MaterialFocusNet')


In [ ]:
# Show MaterialFocusNet's correct and incorrect predictions for qualitative error analysis.
@torch.no_grad()
def collect_predictions(model, dataset, n=12):
    model.eval(); examples = []
    for i in range(len(dataset)):
        image, target = dataset[i]
        pred = model(image.unsqueeze(0).to(DEVICE)).argmax(1).item()
        examples.append((image.cpu(), target, pred))
    correct = [x for x in examples if x[1] == x[2]][:n//2]
    incorrect = [x for x in examples if x[1] != x[2]][:n//2]
    return correct + incorrect

examples = collect_predictions(material_model, test_ds)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, (image, target, pred) in zip(axes.flat, examples):
    view = image.permute(1, 2, 0) * std.numpy() + mean.numpy()
    ax.imshow(view.clip(0, 1)); ax.axis('off')
    ax.set_title(f'True: {class_names[target]}\nPred: {class_names[pred]}', color=('green' if target == pred else 'crimson'))
plt.tight_layout()


## Analysis and conclusion

After running the notebook, write the final report/presentation conclusion from the displayed metrics:

1. State which model achieved the highest **test macro F1** and report its accuracy, macro precision, macro recall, and macro F1.
2. Use both confusion matrices to explain the two most frequent material confusions.
3. Compare the learning curves for overfitting or underfitting.
4. Discuss limitations: TrashNet's small size, class imbalance, controlled backgrounds, and the fact that it contains a single centered item.
5. Future work: test on more realistic waste images, collect additional trash examples, and evaluate mobile/edge deployment.

Do not claim that the attention mechanism helped unless the final test metrics show a meaningful improvement over the baseline.


In [ ]:
# Optional: download the trained checkpoints and notebook outputs from Colab.
from google.colab import files
files.download('baseline_cnn_best.pt')
files.download('material_focus_net_best.pt')
